# Long-short portfolio construction and performance

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


# **CA Models, value-weighed long-short portolio (annualized)**


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    monthly_returns = merged_df.groupby('month').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    performance_df.index = performance_df.index.to_timestamp()  # Convert PeriodIndex to Timestamp

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)  # Annualize the Sharpe ratio
        sharpe_ratios[model_name][n_factors] = sharpe_ratio_annualized
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio_annualized:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Month')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


# **CA Models, equal-weighed long-short portolio (annualized)**


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate equal-weighted decile portfolios
def calculate_deciles(predicted, actual):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_return = top_decile.mean()
    bottom_return = bottom_decile.mean()
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    monthly_returns = merged_df.groupby('month').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    performance_df.index = performance_df.index.to_timestamp()  # Convert PeriodIndex to Timestamp

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)  # Annualize the Sharpe ratio
        sharpe_ratios[model_name][n_factors] = sharpe_ratio_annualized
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio_annualized:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Month')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


# **Benchmark models value-weighted long-short portfolio (annualized)**

##**Loading of the stock_characteristics_monthly_py, which contains stock characteristics data. The data we are using here is the market cap for all stocks to calculalte the value-weighted long-short portfolio**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow scikit-learn joblib tqdm tables

import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())

# Rename the 'month' column to 'date'
df = df.rename(columns={'month': 'date'})

# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['date', 'permno'])

# Sort the MultiIndex
df = df.sort_index()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Filter for dates starting from '1993'
df = df.loc[pd.IndexSlice['1975':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# Define the function to apply quantile transformation for a single date
def process_date(date, df, characteristics):
    group = df.loc[date, characteristics]
    transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
    transformed_df = transformed_df.mul(2).sub(1)
    transformed_df['date'] = date  # Add the date back to the DataFrame
    return transformed_df

# Initialize the progress bar
dates = df.index.get_level_values('date').unique()
pbar = tqdm(dates, desc="Processing Dates")

# Apply the transformation with parallel processing
normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# Concatenate all the normalized data back into a single DataFrame
normalized_df = pd.concat(normalized_data)

# Reset and set the index to match the original DataFrame's multi-index
normalized_df = normalized_df.reset_index().set_index(['date', 'permno'])

# Ensure the index order matches the original DataFrame
normalized_df = normalized_df.reindex(df.index)

# Assign the normalized values back to the original DataFrame
df.loc[:, characteristics] = normalized_df[characteristics].values

# Close the progress bar
pbar.close()

# # Scale the variables
# scaler = StandardScaler()
# df[characteristics] = scaler.fit_transform(df[characteristics])

# Clip the returns columns
df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)

# Print the first few rows of the scaled DataFrame to verify
print(df.head())

# Add 'ret_excess' to the list of columns
columns_to_select = characteristics

# Select these columns from the DataFrame
characteristics_data = df[columns_to_select]

# Define a function to fill missing values with the median for each group
def fill_missing_with_median(group):
    return group.fillna(group.median())

# Apply the function to each group (each date) and preserve the index
df_filled = characteristics_data.groupby(level='date').apply(fill_missing_with_median)

# Remove the additional 'date' level in the index introduced by groupby
df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
df.update(df_filled)

print(df)

dad = df

# Display the updated DataFrame to check the results
print(dad.head())


##**Loading of the prediction data for IPCA, PCA and FF models**




In [ ]:
from sys import pycache_prefix
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
from google.colab import drive
import matplotlib.pyplot as plt

#########################
## LOADING ORIGINAL DF ##
#########################

!pip install tables

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

if 'date' in original_df.columns and 'permno' in original_df.columns:
    original_df.set_index(['date', 'permno'], inplace=True)
else:
    print("The necessary columns 'date' and 'permno' are not present in original_df.")

############################################################
## LOADING AND COMBINING IPCA DATAFRAMES FOR EACH FACTOR ###
############################################################

# Path to the HDF5 file
results_path_ipca = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path_ipca = results_path_ipca / 'predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
ipca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
for n_factors in tqdm(range(1, 7), desc='IPCA'):  # Adjust the range as needed
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'fwd_predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path_ipca, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Store the combined dataframe for the current factor in the dictionary
    ipca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path_ipca / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now combined_dataframes dictionary contains combined dataframes for each factor

##########################################################
## LOADING AND COMBINING PCA DATAFRAMES FOR EACH FACTOR ##
##########################################################

# Path to the HDF5 file
results_path_pca = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_path_pca = results_path_pca / 'predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
pca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PCA'):
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path_pca, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Store the combined dataframe for the current factor in the dictionary
    pca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path_pca / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now combined_dataframes dictionary contains combined dataframes for each factor

###############################################################################################
## RENAMING COLUMNS IN IPCA DF TO y_true and y_pred so they align with the rest of the code ##
###############################################################################################

factors = [1, 2, 3, 4, 5, 6]

for i in factors:
    drassle = ipca_combined_dataframes[i]
    drassle.rename(columns={'actual returns': 'y_true', 'predicted returns': 'y_pred'}, inplace=True)

print(ipca_combined_dataframes)




##**Functions 1**

In [ ]:
##########################################################
## FORMING VALUE-WEIGHTED LONG-SHORT PORTFOLIOS         ##
##########################################################

def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1]
        curr_month = merged_df[merged_df['month'] == date]
        monthly_return = calculate_deciles(prev_month['y_pred'], curr_month['y_true'], curr_month['mktcap_lag'])
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()
    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover


##**Loading of PLS prediction data**

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')


# Path to the saved HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/predictive_results_PLS')
hdf5_path = results_path / 'predictions.h5'

# Function to load and rename predictions from HDF5 for a specific factor
def load_and_process_predictions(hdf5_path, factor):
    try:
        with pd.HDFStore(hdf5_path, mode='r') as store:
            key = f'/pls_predictions/factors_{factor}'
            if key in store:
                predictions = store[key]
                predictions.rename(columns={'actual': 'y_true', 'predicted': 'y_pred'}, inplace=True)
                return predictions
            else:
                raise ValueError(f"No predictions found for {factor} factors in the HDF5 file.")
    except Exception as e:
        print(f"Error loading predictions: {e}")
        return None

# Initialize a dictionary to store combined dataframes for each factor
pls_combined_dataframes = {}

# Load the dataframes for each factor and store them in the dictionary
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PLS'):
    combined_df = load_and_process_predictions(hdf5_path, n_factors)
    if combined_df is not None:
        pls_combined_dataframes[n_factors] = combined_df

# Example: Accessing and displaying the first few rows of the predictions for a specific factor
factor_to_display = 4  # Specify the number of factors you want to display predictions for
if factor_to_display in pls_combined_dataframes:
    print(f"Displaying predictions for {factor_to_display} factors:")
    print(pls_combined_dataframes[factor_to_display].head())
else:
    print(f"No predictions available for {factor_to_display} factors.")

# Now `pls_combined_dataframes` dictionary contains processed dataframes for each factor,
# with columns 'y_true' and 'y_pred' for actual and predicted values, respectively.


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from google.colab import drive
from sklearn.preprocessing import quantile_transform, StandardScaler

#########################
## LOADING ORIGINAL DF ##
#########################
# Mount Google Drive
# drive.mount('/content/drive')
# # Define the file path
# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# # Load the Parquet file into a DataFrame
# df = pd.read_parquet(file_path, engine='pyarrow')

# # Rename the 'month' column to 'date'
# df = df.rename(columns={'month': 'date'})

# # # Ensure 'df' has a MultiIndex ['date', 'permno']
# # df = df.set_index(['date', 'permno'])

# # Sort the MultiIndex
# original_df = df.sort_index()

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade"]

# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Filter for dates starting from '1993'
# df = df.loc[pd.IndexSlice['1975':, :], :]

# # Create forward returns
# df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# # Define the function to apply quantile transformation for a single date
# def process_date(date, df, characteristics):
#     group = df.loc[date, characteristics]
#     transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
#     transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
#     transformed_df = transformed_df.mul(2).sub(1)
#     transformed_df['date'] = date  # Add the date back to the DataFrame
#     return transformed_df

# # Initialize the progress bar
# dates = df.index.get_level_values('date').unique()
# pbar = tqdm(dates, desc="Processing Dates")

# # Apply the transformation with parallel processing
# normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# # Concatenate all the normalized data back into a single DataFrame
# normalized_df = pd.concat(normalized_data)

# # Reset and set the index to match the original DataFrame's multi-index
# normalized_df = normalized_df.reset_index().set_index(['date', 'permno'])

# # Ensure the index order matches the original DataFrame
# normalized_df = normalized_df.reindex(df.index)

# # Assign the normalized values back to the original DataFrame
# df.loc[:, characteristics] = normalized_df[characteristics].values

# # Close the progress bar
# pbar.close()

# # # Scale the variables
# # scaler = StandardScaler()
# # df[characteristics] = scaler.fit_transform(df[characteristics])

# # Clip the returns columns
# df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)

# # Print the first few rows of the scaled DataFrame to verify
# # print(df.head())

# # Add 'ret_excess' to the list of columns
# columns_to_select = characteristics

# # Select these columns from the DataFrame
# characteristics_data = df[columns_to_select]

# # Define a function to fill missing values with the median for each group
# def fill_missing_with_median(group):
#     return group.fillna(group.median())

# # Apply the function to each group (each date) and preserve the index
# df_filled = characteristics_data.groupby(level='date').apply(fill_missing_with_median)

# # Remove the additional 'date' level in the index introduced by groupby
# df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
# df.update(df_filled)

# original_df = df
############################################################
## LOADING AND COMBINING IPCA DATAFRAMES FOR EACH FACTOR ###
############################################################
!pip install tables

# Path to the HDF5 file
results_path_ipca = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path_ipca = results_path_ipca / 'predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
ipca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
for n_factors in tqdm(range(1, 7), desc='IPCA'):  # Adjust the range as needed
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'fwd_predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path_ipca, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Ensure permno column is integer type
    combined_df['permno'] = combined_df['permno'].astype(float).astype(int)

    # Store the combined dataframe for the current factor in the dictionary
    ipca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path_ipca / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now combined_dataframes dictionary contains combined dataframes for each factor

##########################################################
## LOADING AND COMBINING PCA DATAFRAMES FOR EACH FACTOR ##
##########################################################

# Path to the HDF5 file
results_path_pca = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_path_pca = results_path_pca / 'predictions.h5'

# Initialize dictionary to store combined dataframes for each factor
pca_combined_dataframes = {}

# Load the dataframes for each factor and stack them together
factors = [1, 2, 3, 4, 5, 6]  # List of factors
for n_factors in tqdm(factors, desc='PCA'):
    combined_df = pd.DataFrame()  # Initialize an empty dataframe for each factor
    for fold in range(20):  # Assuming 20 folds
        key = f'predictions/n_factors_{n_factors}_fold_{fold}'
        try:
            with pd.HDFStore(hdf5_path_pca, mode='r') as store:
                df = store.get(key)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
        except KeyError:
            print(f"Data for {key} not found.")

    # Ensure permno column is integer type
    combined_df['permno'] = combined_df['permno'].astype(float).astype(int)

    # Store the combined dataframe for the current factor in the dictionary
    pca_combined_dataframes[n_factors] = combined_df

    # Save the combined dataframe for each factor if needed
    combined_hdf5_path = results_path_pca / f'combined_predictions_n_factors_{n_factors}.h5'
    with pd.HDFStore(combined_hdf5_path, mode='w', complevel=9, complib='blosc:zstd') as store:
        store.put(f'combined_predictions/n_factors_{n_factors}', combined_df, format='table')

# Now combined_dataframes dictionary contains combined dataframes for each factor

###############################################################################################
## RENAMING COLUMNS IN IPCA DF TO y_true and y_pred so they align with the rest of the code ##
###############################################################################################

factors = [1, 2, 3, 4, 5, 6]

for i in factors:
    drassle = ipca_combined_dataframes[i]
    drassle.rename(columns={'actual returns': 'y_true', 'predicted returns': 'y_pred'}, inplace=True)

print(ipca_combined_dataframes)

##############################
## LOADING FF MODEL RETURNS ##
##############################
import pandas as pd
from pathlib import Path

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
hdf5_path = results_path / 'fwd_predictions.h5'  # Adjusted to match the correct file name used during saving

# Factors sequences to use (same as used during saving)
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW','Mom']
]

# Dictionary to hold the concatenated predictions
concatenated_predictions = {}

# Load the predictions from the HDF5 file
with pd.HDFStore(hdf5_path, mode='r') as store:
    # List all keys in the store
    all_keys = store.keys()
    print(f"All keys in the store: {all_keys}")

    for n_factors in range(1, len(factor_sequences) + 1):
        factor_dfs = []
        for fold_key in all_keys:
            if f'/predictions/n_factors_{n_factors}_fold_' in fold_key:
                print(f"Loading key: {fold_key}")
                factor_df = store[fold_key]
                factor_dfs.append(factor_df)

        if factor_dfs:
            # Concatenate all folds for the current factor sequence
            concatenated_predictions[n_factors] = pd.concat(factor_dfs, ignore_index=True)
            print(f"Concatenated {len(factor_dfs)} dataframes for {n_factors} factors.")
        else:
            print(f"No dataframes found for n_factors_{n_factors}")

# Example: Access the concatenated DataFrame for the sequence of 6 factors
if 6 in concatenated_predictions:
    df_six_factors = concatenated_predictions[6]
    print(f"Head of concatenated DataFrame for 6 factors:\n{df_six_factors.head()}")
else:
    print("No concatenated predictions found for 6 factors.")





##**Functions 2 in use Returns metrics and plots**

In [ ]:

##########################################################
## FORMING VALUE-WEIGHTED LONG-SHORT PORTFOLIOS         ##
##########################################################
def calculate_deciles(predicted, actual):
    # Ensure the indices are aligned
    predicted, actual = predicted.align(actual, join='inner')

    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_return = top_decile.mean()
    bottom_return = bottom_decile.mean()
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

def evaluate_performance(predictions_df, original_df):
    # Ensure the columns being merged on have the same data types
    original_df['date'] = pd.to_datetime(original_df.index.get_level_values('date'))

    # Ensure 'permno' is integer in original_df
    original_df['permno'] = original_df.index.get_level_values('permno').astype(int)

    # Ensure 'permno' in predictions_df is integer
    predictions_df.index = predictions_df.index.set_levels([
        predictions_df.index.levels[0],
        predictions_df.index.levels[1].astype(float).astype(int)
    ], level=[0, 1])

    # Set index for merging if not already set in predictions_df
    if not predictions_df.index.names == ['date', 'permno']:
        predictions_df.set_index(['date', 'permno'], inplace=True)

    # Perform the join operation
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred' and 'y_true' if any
    merged_df.dropna(subset=['y_pred', 'y_true'], inplace=True)

    # Create 'month' column
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted and actual values
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns
        monthly_return = calculate_deciles(prev_month['y_pred'], curr_month['y_true'])
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover

# Evaluate models with 1 to 6 factors
metrics = {
    'sharpe_ratios': {},
    'avg_returns': {},
    'volatilities': {},
    'vars_95': {},
    'turnovers': {}
}

for factor in factors:
    predictions_df = concatenated_predictions[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for FF with $K$={factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


for factor in factors:
    predictions_df = ipca_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for IPCA with {factor} Factors')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


for factor in factors:
    predictions_df = pca_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factor ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


In [ ]:
def calculate_deciles(predicted, actual, weights):
    # Ensure the indices are aligned by combining into a DataFrame
    data = pd.concat([predicted, actual, weights], axis=1, join='inner')
    data.dropna(inplace=True)

    # Debugging statements
    print("Data after concatenation and dropping NA:")
    print(data.head())
    print()

    # Extract aligned Series
    predicted = data.iloc[:, 0]
    actual = data.iloc[:, 1]
    weights = data.iloc[:, 2]

    # Assign deciles based on the ranked predicted values
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1

    # Debugging statements
    print("Deciles assigned:")
    print(deciles.head())
    print()

    # Select actual returns and weights for the top and bottom deciles
    top_decile_mask = deciles == 10
    bottom_decile_mask = deciles == 1

    top_decile_returns = actual[top_decile_mask]
    top_decile_weights = weights[top_decile_mask]
    bottom_decile_returns = actual[bottom_decile_mask]
    bottom_decile_weights = weights[bottom_decile_mask]

    # Debugging statements
    print("Top decile returns and weights:")
    print(pd.DataFrame({'Return': top_decile_returns, 'Weight': top_decile_weights}).head())
    print()
    print("Bottom decile returns and weights:")
    print(pd.DataFrame({'Return': bottom_decile_returns, 'Weight': bottom_decile_weights}).head())
    print()

    # Normalize weights within each decile so they sum to 1
    top_weights_normalized = top_decile_weights / top_decile_weights.sum()
    bottom_weights_normalized = bottom_decile_weights / bottom_decile_weights.sum()

    # Debugging statements
    print("Normalized top decile weights:")
    print(top_weights_normalized.head())
    print()
    print("Normalized bottom decile weights:")
    print(bottom_weights_normalized.head())
    print()

    # Calculate value-weighted returns for top and bottom deciles
    top_return = (top_decile_returns * top_weights_normalized).sum()
    bottom_return = (bottom_decile_returns * bottom_weights_normalized).sum()

    # Debugging statements
    print(f"Top decile return: {top_return}")
    print(f"Bottom decile return: {bottom_return}")
    print()

    # Compute the zero-net investment return
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return


def evaluate_performance(predictions_df, original_df):
    # Ensure the columns being merged on have the same data types
    original_df['date'] = pd.to_datetime(original_df.index.get_level_values('date'))

    # Ensure 'permno' is integer in original_df
    original_df['permno'] = original_df.index.get_level_values('permno').astype(int)

    # Check if predictions_df has 'date' and 'permno' as a MultiIndex
    if not isinstance(predictions_df.index, pd.MultiIndex) or predictions_df.index.names != ['date', 'permno']:
        print("Predictions DataFrame index is not a MultiIndex. Creating MultiIndex using 'date' and 'permno' columns.")

        # Assuming 'date' and 'permno' are columns in predictions_df
        if 'date' in predictions_df.columns and 'permno' in predictions_df.columns:
            predictions_df['date'] = pd.to_datetime(predictions_df['date'])
            # Convert 'permno' to float first, then to int
            try:
                predictions_df['permno'] = predictions_df['permno'].astype(float).astype(int)
            except ValueError as e:
                print("Error converting 'permno' to integer:", e)
                return
            predictions_df.set_index(['date', 'permno'], inplace=True)
        else:
            print("Error: 'date' and 'permno' columns are missing in predictions_df.")
            return

    # Perform the join operation to include 'mktcap_lag' for weighting
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred', 'y_true', and 'mktcap_lag' if any
    merged_df.dropna(subset=['y_pred', 'y_true', 'mktcap_lag'], inplace=True)

    # Create 'month' column and sort by date
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted, actual values, and weights
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns using value-weighted method
        monthly_return = calculate_deciles(
            prev_month['y_pred'],
            curr_month['y_true'],
            prev_month['mktcap_lag']
        )
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover

    # Print the index of predictions_df after setting the MultiIndex
    print("Predictions DataFrame index after setting MultiIndex:")
    print(predictions_df.index)
    print()

    # Perform the join operation
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Print the merged dataframe
    print("Merged DataFrame:")
    print(merged_df.head())
    print()

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred' and 'y_true' if any
    merged_df.dropna(subset=['y_pred', 'y_true'], inplace=True)

    # Print after handling missing values
    print("Merged DataFrame after dropping NA values:")
    print(merged_df.head())
    print()

    # Continue with the rest of the function...
    # Create 'month' column and other calculations
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted and actual values
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns
        monthly_return = calculate_deciles(prev_month['y_pred'], curr_month['y_true'])
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover


# Evaluate models with 1 to 6 factors
metrics = {
    'sharpe_ratios': {},
    'avg_returns': {},
    'volatilities': {},
    'vars_95': {},
    'turnovers': {}
}

for factor in factors:
    predictions_df = concatenated_predictions[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for FF with $K$={factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


for factor in factors:
    predictions_df = ipca_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for IPCA with {factor} Factors')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


for factor in factors:
    predictions_df = pca_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factor ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


# **Actual Value Weighted portfolio**

In [ ]:
def calculate_deciles(predicted, actual, weights):
    # Ensure the indices are aligned by combining into a DataFrame
    data = pd.concat([predicted, actual, weights], axis=1, join='inner')
    data.dropna(inplace=True)

    # Extract aligned Series
    predicted = data.iloc[:, 0]
    actual = data.iloc[:, 1]
    weights = data.iloc[:, 2]

    # Assign deciles based on the ranked predicted values
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1

    # Select actual returns and weights for the top and bottom deciles
    top_decile_mask = deciles == 10
    bottom_decile_mask = deciles == 1

    top_decile_returns = actual[top_decile_mask]
    top_decile_weights = weights[top_decile_mask]
    bottom_decile_returns = actual[bottom_decile_mask]
    bottom_decile_weights = weights[bottom_decile_mask]

    # Normalize weights within each decile so they sum to 1
    top_weights_normalized = top_decile_weights / top_decile_weights.sum()
    bottom_weights_normalized = bottom_decile_weights / bottom_decile_weights.sum()

    # Calculate value-weighted returns for top and bottom deciles
    top_return = (top_decile_returns * top_weights_normalized).sum()
    bottom_return = (bottom_decile_returns * bottom_weights_normalized).sum()

    # Compute the zero-net investment return
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return


def evaluate_performance(predictions_df, original_df):
    # Ensure the columns being merged on have the same data types
    original_df['date'] = pd.to_datetime(original_df.index.get_level_values('date'))

    # Ensure 'permno' is integer in original_df
    original_df['permno'] = original_df.index.get_level_values('permno').astype(int)

    # Check if predictions_df has 'date' and 'permno' as a MultiIndex
    if not isinstance(predictions_df.index, pd.MultiIndex) or predictions_df.index.names != ['date', 'permno']:
        # Assuming 'date' and 'permno' are columns in predictions_df
        if 'date' in predictions_df.columns and 'permno' in predictions_df.columns:
            predictions_df['date'] = pd.to_datetime(predictions_df['date'])
            # Convert 'permno' to float first, then to int
            try:
                predictions_df['permno'] = predictions_df['permno'].astype(float).astype(int)
            except ValueError as e:
                print("Error converting 'permno' to integer:", e)
                return
            predictions_df.set_index(['date', 'permno'], inplace=True)
        else:
            print("Error: 'date' and 'permno' columns are missing in predictions_df.")
            return

    # Perform the join operation to include 'mktcap_lag' for weighting
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred', 'y_true', and 'mktcap_lag' if any
    merged_df.dropna(subset=['y_pred', 'y_true', 'mktcap_lag'], inplace=True)

    # Create 'month' column and sort by date
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted, actual values, and weights
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns using value-weighted method
        monthly_return = calculate_deciles(
            prev_month['y_pred'],
            curr_month['y_true'],
            prev_month['mktcap_lag']
        )
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())

    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover


# Evaluate models with specified factors
metrics = {
    'sharpe_ratios': {},
    'avg_returns': {},
    'volatilities': {},
    'vars_95': {},
    'turnovers': {}
}

# Assuming 'factors' is a list of factor counts you want to evaluate
for factor in factors:
    predictions_df = concatenated_predictions[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for FF with $K$={factor} Factors')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

for factor in factors:
    predictions_df = ipca_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for IPCA with {factor} Factors')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

for factor in factors:
    predictions_df = pca_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factors')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


In [ ]:
for factor in factors:
    predictions_df = pls_combined_dataframes[factor]
    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['sharpe_ratios'][factor] = sharpe_ratio
    metrics['avg_returns'][factor] = avg_return
    metrics['volatilities'][factor] = volatility
    metrics['vars_95'][factor] = var_95
    metrics['turnovers'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factors')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


##**Functions 3 in use Returns Tables and plots**

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

def calculate_deciles(predicted, actual):
    # Ensure the indices are aligned
    predicted, actual = predicted.align(actual, join='inner')

    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_return = top_decile.mean()
    bottom_return = bottom_decile.mean()
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

def evaluate_performance(predictions_df, original_df):
    # Ensure the columns being merged on have the same data types
    # original_df['date'] = pd.to_datetime(original_df.index.get_level_values('date'))

    # Ensure 'permno' is integer in original_df
    # original_df['permno'] = original_df.index.get_level_values('permno').astype(int)

    # Ensure 'permno' in predictions_df is integer
    predictions_df.index = predictions_df.index.set_levels([
        predictions_df.index.levels[0],
        predictions_df.index.levels[1].astype(float).astype(int)
    ], level=[0, 1])

    # Set index for merging if not already set in predictions_df
    if not predictions_df.index.names == ['date', 'permno']:
        predictions_df.set_index(['date', 'permno'], inplace=True)

    # Perform the join operation
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred' and 'y_true' if any
    merged_df.dropna(subset=['y_pred', 'y_true'], inplace=True)

    # Create 'month' column
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted and actual values
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns
        monthly_return = calculate_deciles(prev_month['y_pred'], curr_month['y_true'])
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover

# Evaluate models with 1 to 6 factors
metrics = {
    'Sharpe Ratio': {},
    'Average Return': {},
    'Volatility': {},
    'Value-At-Risk (95%)': {},
    'Turnover': {}
}

for factor in range(1, 7):
    predictions_df = pls_combined_dataframes[factor]
    predictions_df.set_index(['date', 'permno'], inplace=True)

    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factor ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)


##**with R^2**

In [ ]:
original_df

##**PLS functions 3**

In [ ]:

for factor in range(1, 7):
    predictions_df = pls_combined_dataframes[factor]
    # predictions_df.set_index(['date', 'permno'], inplace=True)

    performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factor ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)


In [ ]:
pls_combined_dataframes[1]

In [ ]:
original_df

In [ ]:
print(original_df.index.names)  # Check the names of the index levels
print(original_df.columns)      # Check the column names


In [ ]:
if 'date' in original_df.columns and 'permno' in original_df.columns:
    original_df.set_index(['date', 'permno'], inplace=True)
else:
    print("The necessary columns 'date' and 'permno' are not present in original_df.")


In [ ]:
predictions_df.index

In [ ]:
print(predictions_df)

In [ ]:
predictions_df.index = predictions_df.index.set_names(['date', 'permno'])


#**Extra Performance Metrics VALUE WEIGHTED**

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

def calculate_deciles(predicted, actual):
    # Ensure the indices are aligned
    predicted, actual = predicted.align(actual, join='inner')

    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_return = top_decile.mean()
    bottom_return = bottom_decile.mean()
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())

    # Calculate maximum drawdown
    cumulative_returns = (1 + returns).cumprod() - 1
    rolling_max = cumulative_returns.cummax()
    drawdown = (rolling_max - cumulative_returns) / rolling_max
    max_drawdown = drawdown.max()

    # Calculate maximum loss for one month
    max_loss = returns.min()

    return avg_return, volatility, var_95, turnover, max_drawdown, max_loss

def calculate_r_squared(actual, predicted):
    correlation_matrix = np.corrcoef(actual, predicted)
    correlation_xy = correlation_matrix[0, 1]
    r_squared = correlation_xy**2
    return r_squared

def calculate_information_ratio(returns, benchmark_returns):
    active_return = returns - benchmark_returns
    tracking_error = active_return.std()
    information_ratio = active_return.mean() / tracking_error
    return information_ratio

def evaluate_performance(predictions_df, original_df):
    # Ensure the columns being merged on have the same data types
    original_df['date'] = pd.to_datetime(original_df.index.get_level_values('date'))

    # Ensure 'permno' is integer in original_df
    original_df['permno'] = original_df.index.get_level_values('permno').astype(int)

    # Ensure 'permno' in predictions_df is integer
    predictions_df.index = predictions_df.index.set_levels([
        predictions_df.index.levels[0],
        predictions_df.index.levels[1].astype(float).astype(int)
    ], level=[0, 1])

    # Set index for merging if not already set in predictions_df
    if not predictions_df.index.names == ['date', 'permno']:
        predictions_df.set_index(['date', 'permno'], inplace=True)

    # Perform the join operation
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred' and 'y_true' if any
    merged_df.dropna(subset=['y_pred', 'y_true'], inplace=True)

    # Create 'month' column
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted and actual values
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns
        monthly_return = calculate_deciles(prev_month['y_pred'], curr_month['y_true'])
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover, max_drawdown, max_loss = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    # Calculate R^2 and Information Ratio
    r_squared = calculate_r_squared(merged_df['y_true'], merged_df['y_pred'])
    information_ratio = calculate_information_ratio(performance_df['Zero_Net_Investment_Returns'], merged_df['y_true'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, max_drawdown, max_loss, r_squared, information_ratio

# Evaluate models with 1 to 6 factors
metrics = {
    'Sharpe Ratio': {},
    'Average Return': {},
    'Volatility': {},
    'Value-At-Risk (95%)': {},
    'Turnover': {},
    'Max Drawdown': {},
    'Max Loss': {},
    'R-squared': {}
    #, 'Information Ratio': {}
}



for factor in range(1, 7):
    predictions_df = ipca_combined_dataframes[factor]
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared
    # metrics['Information Ratio'][factor] = information_ratio

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    # print("Information Ratio:", information_ratio)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for IPCA with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)



for factor in range(1, 7):
    predictions_df = pca_combined_dataframes[factor]
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared
    # metrics['Information Ratio'][factor] = information_ratio

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    # print("Information Ratio:", information_ratio)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCA with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)




for factor in range(1, 7):
    predictions_df = concatenated_predictions[factor]
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared
    # metrics['Information Ratio'][factor] = information_ratio

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    # print("Information Ratio:", information_ratio)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for FF model with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)


In [ ]:


print(pca_combined_dataframes[1].index)
print(pca_combined_dataframes[1].columns)

In [ ]:
ipca_combined_dataframes[1].index

In [ ]:

# Example evaluation for a specific IPCA model and factor
model_name = 'IPCA'
factor = 1  # Example factor
predictions_df = ipca_combined_dataframes[factor]
performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

# Print metrics
print("Sharpe Ratio:", sharpe_ratio)
print("Average Return:", avg_return)
print("Volatility:", volatility)
print("Value-At-Risk (95%):", var_95)
print("Turnover:", turnover)

# Plot cumulative returns
performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for {model_name} Factor {factor}')
plt.xlabel('Month')
plt.ylabel('Cumulative Returns')
plt.show()

# **Benchmark models equal-weighted long-short portfolio (annualized)**

##**The FF model was not working, because it was not defined**

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

def calculate_deciles(predicted, actual):
    # Ensure the indices are aligned
    predicted, actual = predicted.align(actual, join='inner')

    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]

    # Calculate equal-weighted returns for top and bottom deciles
    top_return = top_decile.mean()
    bottom_return = bottom_decile.mean()
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())

    # Calculate maximum drawdown
    cumulative_returns = (1 + returns).cumprod() - 1
    rolling_max = cumulative_returns.cummax()
    drawdown = (rolling_max - cumulative_returns) / rolling_max
    max_drawdown = drawdown.max()

    # Calculate maximum loss for one month
    max_loss = returns.min()

    return avg_return, volatility, var_95, turnover, max_drawdown, max_loss

def calculate_r_squared(actual, predicted):
    correlation_matrix = np.corrcoef(actual, predicted)
    correlation_xy = correlation_matrix[0, 1]
    r_squared = correlation_xy**2
    return r_squared

def calculate_information_ratio(returns, benchmark_returns):
    active_return = returns - benchmark_returns
    tracking_error = active_return.std()
    information_ratio = active_return.mean() / tracking_error
    return information_ratio

def evaluate_performance(predictions_df, original_df):
    # Ensure the columns being merged on have the same data types
    original_df['date'] = pd.to_datetime(original_df.index.get_level_values('date'))

    # Ensure 'permno' is integer in original_df
    original_df['permno'] = original_df.index.get_level_values('permno').astype(int)

    # Ensure 'permno' in predictions_df is integer
    predictions_df.index = predictions_df.index.set_levels([
        predictions_df.index.levels[0],
        predictions_df.index.levels[1].astype(float).astype(int)
    ], level=[0, 1])

    # Set index for merging if not already set in predictions_df
    if not predictions_df.index.names == ['date', 'permno']:
        predictions_df.set_index(['date', 'permno'], inplace=True)

    # Perform the join operation
    merged_df = predictions_df.join(original_df[['mktcap_lag']], how='inner')

    # Reset the index after the merge
    merged_df.reset_index(inplace=True)

    # Handle missing values in 'y_pred' and 'y_true' if any
    merged_df.dropna(subset=['y_pred', 'y_true'], inplace=True)

    # Create 'month' column
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    merged_df = merged_df.sort_values(by='date')

    monthly_returns = []
    for date in merged_df['month'].unique()[1:]:  # Skip the first month as it has no t-1 predictions
        prev_month = merged_df[merged_df['month'] == date - 1].copy()
        curr_month = merged_df[merged_df['month'] == date].copy()

        # Ensure the indices are aligned for prev_month and curr_month
        prev_month.set_index('permno', inplace=True)
        curr_month.set_index('permno', inplace=True)

        # Align the indices of predicted and actual values
        prev_month, curr_month = prev_month.align(curr_month, join='inner', axis=0)

        # Calculate monthly returns
        monthly_return = calculate_deciles(prev_month['y_pred'], curr_month['y_true'])
        monthly_returns.append(monthly_return)

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    }, index=merged_df['month'].unique()[1:].to_timestamp())
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    # Calculate the annualized Sharpe ratio
    mean_return = performance_df['Zero_Net_Investment_Returns'].mean()
    std_dev_return = performance_df['Zero_Net_Investment_Returns'].std()
    sharpe_ratio = mean_return / std_dev_return
    annualized_sharpe_ratio = sharpe_ratio * np.sqrt(12)

    avg_return, volatility, var_95, turnover, max_drawdown, max_loss = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    # Calculate R^2 and Information Ratio
    r_squared = calculate_r_squared(merged_df['y_true'], merged_df['y_pred'])
    information_ratio = calculate_information_ratio(performance_df['Zero_Net_Investment_Returns'], merged_df['y_true'])

    return performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, max_drawdown, max_loss, r_squared, information_ratio

# Evaluate models with 1 to 6 factors
metrics = {
    'Sharpe Ratio': {},
    'Average Return': {},
    'Volatility': {},
    'Value-At-Risk (95%)': {},
    'Turnover': {},
    'Max Drawdown': {},
    'Max Loss': {},
    'R-squared': {}
    #, 'Information Ratio': {}
}

for factor in range(1, 7):
    predictions_df = ipca_combined_dataframes[factor]
    # Do not set the index again since it's already set
    # predictions_df.set_index(['date', 'permno'], inplace=True)
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, \
    max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for IPCA with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)


for factor in range(1, 7):
    predictions_df = pca_combined_dataframes[factor]
    # Do not set the index again since it's already set
    # predictions_df.set_index(['date', 'permno'], inplace=True)
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, \
    max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PCR with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)


for factor in range(1, 7):
    predictions_df = concatenated_predictions[factor]
    # Do not set the index again since it's already set
    # predictions_df.set_index(['date', 'permno'], inplace=True)
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, \
    max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for FF with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)


for factor in range(1, 7):
    predictions_df = pls_combined_dataframes[factor]
    # Do not set the index again since it's already set
    # predictions_df.set_index(['date', 'permno'], inplace=True)
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, \
    max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PLS with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()


# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)



In [ ]:
print(predictions_df.head())
print(predictions_df.columns)
print(predictions_df.index.names)


##**equal-weighted portfolio performance evaluation PLS**

In [ ]:
for factor in range(1, 7):
    predictions_df = pls_combined_dataframes[factor]
    predictions_df.set_index(['date', 'permno'], inplace=True)
    performance_df, annualized_sharpe_ratio, avg_return, volatility, var_95, turnover, max_drawdown, max_loss, r_squared, information_ratio = evaluate_performance(predictions_df, original_df)

    # Store metrics in the dictionary
    metrics['Sharpe Ratio'][factor] = annualized_sharpe_ratio
    metrics['Average Return'][factor] = avg_return
    metrics['Volatility'][factor] = volatility
    metrics['Value-At-Risk (95%)'][factor] = var_95
    metrics['Turnover'][factor] = turnover
    metrics['Max Drawdown'][factor] = max_drawdown
    metrics['Max Loss'][factor] = max_loss
    metrics['R-squared'][factor] = r_squared
    # metrics['Information Ratio'][factor] = information_ratio

    # Print metrics
    print(f"Factor {factor}:")
    print("Sharpe Ratio:", annualized_sharpe_ratio)
    print("Average Return:", avg_return)
    print("Volatility:", volatility)
    print("Value-At-Risk (95%):", var_95)
    print("Turnover:", turnover)
    print("Max Drawdown:", max_drawdown)
    print("Max Loss:", max_loss)
    print("R-squared:", r_squared)
    # print("Information Ratio:", information_ratio)
    print()

    # Plot cumulative returns
    performance_df['Cumulative_Returns'].plot(title=f'Cumulative Returns for PLS model with {factor} Factors ')
    plt.xlabel('Month')
    plt.ylabel('Cumulative Returns')
    plt.show()

# Convert the metrics dictionary to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics)

# Generate LaTeX tables
latex_table = metrics_df.to_latex()

print("Metrics in LaTeX table format:")
print(latex_table)

